# Lecture 7: Operations on Arithmetic Expressions
Originally by Sriram Sankaranarayanan <srirams@colorado>

Modified by Ravi Mangal <ravi.mangal@colostate>

Last Modified: Feb 13, 2025.

In this lecture, we will study some opeartions on the inductively defined arithmetic expressions. First, let us look at the grammar defining arithmetic expressions

## Arithmetic Expressions



Arithmetic expressions are generated by the grammar we saw before.

$$\begin{array}{rcll}
\textbf{Expr} & \rightarrow & Const(\textbf{Double}) & \\
& |  & Ident(\textbf{Identifier}) & \\
& | & Plus( \textbf{Expr}^+) & \text{Note:}\ A^+ \ \text{is one or more reps of}\ A \\
& | & Minus( \textbf{Expr}, \textbf{Expr}^+) & \\
& | & Mult(\textbf{Expr}^+) & \\
& | & Div(\textbf{Expr}, \textbf{Expr}) & \\
& | & Log(\textbf{Expr}) & \\
& | & Exp(\textbf{Expr}) & \\
& | & Sine(\textbf{Expr}) & \\
& | & Cosine(\textbf{Expr}) &\\\\
\textbf{Double} & \rightarrow & \text{all double precision numbers in Scala} & \\
\textbf{Identifier} & \rightarrow & [a-zA-Z][a-z\ A-Z\ 0-9\ \_]* & \text{Note: All strings that begin with letters}\\
&&& \text{a-z or A-Z and subsequently can contain a-z, A-Z, 0-9 or \_ chars}
\end{array}$$

In [4]:
sealed trait Expr
case class Const(f: Double) extends Expr 
case class Ident(s: String) extends Expr // We allow any string to be an identifier for now 
                                         // instead of the regular expression shown in the grammar.
case class Plus( eList: List[Expr] ) extends Expr
case class Minus(e1: Expr, eList: List[Expr]) extends Expr
case class Mult(eList: List[Expr]) extends Expr
case class Div(e1: Expr, e2: Expr) extends Expr
case class Log(e: Expr) extends Expr
case class Exp(e: Expr) extends Expr
case class Sine(e: Expr) extends Expr
case class Cosine(e: Expr) extends Expr

defined trait Expr
defined class Const
defined class Ident
defined class Plus
defined class Minus
defined class Mult
defined class Div
defined class Log
defined class Exp
defined class Sine
defined class Cosine

The simplest function we wish to write will collect all the variables that are used in a given expression inside a Set data structure. Read about Scala sets here: https://docs.scala-lang.org/overviews/collections/sets.html



In [5]:
def collectAllVariables(e: Expr): Set[String] = {
    def helperFunction(eList: List[Expr], startValue: Set[String]): Set[String] = {
        // Iterate through the list eList and collect set of variables for each expression in eList.
        // Next: take the union of all the sets with the given set startValue.

        // A nifty functional way to do this
            eList.foldLeft (startValue) {
                case (setSoFar: Set[String], newExpr:Expr) => { val v = collectAllVariables(newExpr)
                                              setSoFar union v  // Same as setSoFar.union(v); Lookup Scala's infix notation 
                                                                // (https://docs.scala-lang.org/style/method-invocation.html)
                                            }
        
            }
        
        // If you insist on a for loop, you have to use a var
        // var setSoFar = startValue
        // for (newExpr <- eList) {
        //   setSoFar = setSoFar.union(collectAllVariables(newExpr))
        //    }
        // return setSoFar
    }
        
    e match {
        case Const(f) => Set() // Return the empty set
        case Ident(s) => Set(s) // Return a singleton set with s in it.
        case Plus(eList) => helperFunction(eList, Set())  //Union of all variables in eList
        case Mult(eList) => helperFunction(eList, Set()) // Union of all variables in eList
    
        case Minus(e1, eList) => {
            val e1Set = collectAllVariables(e1) // First collect variables in e1
            helperFunction(eList, e1Set) // take union of all variables in eList with e1Set from prev. step
        }
    
    
        case Div(e1, e2) => {
            (collectAllVariables(e1)) union (collectAllVariables(e2))
        }
        case Log(e1) => collectAllVariables(e1)
        case Sine(e1) => collectAllVariables(e1)
        case Cosine(e1) => collectAllVariables(e1)
        case Exp(e1) => collectAllVariables(e1)
        case _ => { assert(false); return Set()  } // This is the catch all. Should not happen but good coding practice
    }
}  

defined function collectAllVariables

In [6]:
// cos(x) + sin(y) + e^{x - y - Zzz}
val x = Ident("x")
val y = Ident("y")
val z = Ident("Zzz")
val expr1 = Plus(List(Cosine(x), Sine(y), Exp(Minus(x, List(y,z)))))

x: Ident = Ident(s = "x")
y: Ident = Ident(s = "y")
z: Ident = Ident(s = "Zzz")
expr1: Plus = Plus(
  eList = List(
    Cosine(e = Ident(s = "x")),
    Sine(e = Ident(s = "y")),
    Exp(
      e = Minus(
        e1 = Ident(s = "x"),
        eList = List(Ident(s = "y"), Ident(s = "Zzz"))
      )
    )
  )
)

In [7]:
val s1 = collectAllVariables(expr1)

s1: Set[String] = Set("x", "y", "Zzz")

In [8]:
val expr2 = Plus(List(Exp(Const(1.5)), Sine(Const(2.0)), Cosine(Mult(List(Const(1.2), Const(2.4))))))

expr2: Plus = Plus(
  eList = List(
    Exp(e = Const(f = 1.5)),
    Sine(e = Const(f = 2.0)),
    Cosine(e = Mult(eList = List(Const(f = 1.2), Const(f = 2.4))))
  )
)

In [9]:
val s2 = collectAllVariables(expr2)

s2: Set[String] = Set()

Another important function is pretty print. We would like to pretty print our expression to Latex so that we can cut and paste it in our documents.

Latex allows you to type formulae and have them displayed. Eg., let us take the formula

$$ \frac{M c^2}{ 1 + \frac{1}{1 + c^2}}$$

You would write it as

\$ \\frac{M c^2}{ 1 + \\frac{1}{1 + c^2} \$

In [10]:
def toLatex (e: Expr): String = {
    e match {
        case Const(f) => "%2.2f".format(f) // easy way to convert f to a string.
        case Ident(str) => str
        case Plus(eList) => { val strList = eList map { e =>  toLatex(e) }
                         "\\left("+ strList.mkString (" + ") + "\\right)" // Join each expression into a string separated by " + "
                       }
        case Minus(e, eList) => {
            val s1 = toLatex(e)
            val strList = eList map {e => toLatex(e)} 
             "\\left("+ s1 + " - " + (strList mkString " - ")+ "\\right)" 
            
        }
        case Mult(eList) => {  "\\left(" + (eList map {toLatex(_)} mkString ("\\ ")) + "\\right)"  }
        
        case Div(e1, e2) => { "\\frac{" + (toLatex(e1)) + "}{" + (toLatex(e2)) + "}" }
        case Log(e) => { "\\log(" + (toLatex(e)) + ")" }
        case Exp(e) => { "e^{" + (toLatex(e)) + "}"}
        case Sine(e) => { "\\sin\\left("+ toLatex(e) + "\\right)" }
        case Cosine(e) => {"\\cos\\left("+ toLatex(e) + "\\right)" }
    }
    
}

defined function toLatex

In [11]:
val str1= "$"+toLatex(expr1)+"$"
print(str1)
latex(str1)

$\left(\cos\left(x\right) + \sin\left(y\right) + e^{\left(x - y - Zzz\right)}\right)$

str1: String = "$\\left(\\cos\\left(x\\right) + \\sin\\left(y\\right) + e^{\\left(x - y - Zzz\\right)}\\right)$"

In [12]:
val w = Ident("w")
val expr2 = Div(Minus( Plus(List(w, Mult(List(x, y)), 
                                Plus(List(Const(1.2), w))) ), 
                                     List(Const(1.5), w, x) ), Const(2.3))
latex("$"+ toLatex(expr2) + "$")


w: Ident = Ident(s = "w")
expr2: Div = Div(
  e1 = Minus(
    e1 = Plus(
      eList = List(
        Ident(s = "w"),
        Mult(eList = List(Ident(s = "x"), Ident(s = "y"))),
        Plus(eList = List(Const(f = 1.2), Ident(s = "w")))
      )
    ),
    eList = List(Const(f = 1.5), Ident(s = "w"), Ident(s = "x"))
  ),
  e2 = Const(f = 2.3)
)

### My First Eval Function

Now we will write a very important operation called `eval`. This operation is the primary operation that an interpreter needs to carry out. 

But we will start small by writing an eval for our simple expression grammar. Before we do that, let us ask ourselves two questions: (a) what does it mean for the expressions to be evaluated? and (b) what is the missing information that we need before we can evaluate our arithmetic expressions defined above?
The answers should be easy in this case:

1. An arithmetic expression stands for a number which is the answer when we calculate it and 
2. Of course, the missing information is that we do not know what the valuations for the variables are to start with.

Thus, we are ready to define the eval function.

__Environment__ 

An environment $\eta$ maps every identifier string to a numeric value. 


Example: 

$$ \{ x \mapsto 2, y \mapsto 1.5, Zzz \mapsto 2.8, w \mapsto 15.2 \}$$

Scala has a Map data structure that is ideal for environments

In [50]:
val myEnvironment: Map[String, Double] = Map("x" -> 2.0, "y" -> 1.5, "Zzz" -> 2.8, "w" -> 15.2, "l" -> 129.3)

myEnvironment: Map[String, Double] = HashMap(
  "x" -> 2.0,
  "y" -> 1.5,
  "Zzz" -> 2.8,
  "l" -> 129.3,
  "w" -> 15.2
)

In [51]:
def evalExpr (e: Expr, env: Map[String, Double]): Double = e match {
    case Const (f) => f
    case Ident (str) => { if (env.contains(str)){
                                env(str)
                            } else {
        throw new IllegalArgumentException(s"Environment does not contain mapping for $str")
    }
                        }
    case Plus(eList) => {
        (eList map {evalExpr(_, env)}).sum
    }
    
    case Minus(e, eList) => {
        (evalExpr(e, env)) - ((eList map {evalExpr(_, env)}).sum )
    }
    
    case Mult(eList) => {
        (eList map {evalExpr(_, env)}).product
    }
    
    case Div(e1, e2) => {
        (evalExpr(e1, env)) / (evalExpr(e2, env))
    }
    
    case Log(e) => math.log(evalExpr(e, env))
    
    
    case Exp(e) => math.exp( evalExpr(e, env))
    
    case Sine(e) => math.sin( evalExpr(e, env))
    
    case Cosine(e) => math.cos(evalExpr(e, env))
}

defined function evalExpr

In [52]:
val f1 = evalExpr(expr1, myEnvironment)

f1: Double = 0.6816069937797158

In [53]:
val f2 = evalExpr(expr2, myEnvironment)

f2: Double = 6.913043478260868

In [54]:
val expr3 = Minus(Const(1.0), List(
    Mult(List(Sine(x), Sine(x))),
    Mult(List(Cosine(x), Cosine(x)))
))

latex("$"+toLatex(expr3)+"$")

expr3: Minus = Minus(
  e1 = Const(f = 1.0),
  eList = List(
    Mult(eList = List(Sine(e = Ident(s = "x")), Sine(e = Ident(s = "x")))),
    Mult(eList = List(Cosine(e = Ident(s = "x")), Cosine(e = Ident(s = "x"))))
  )
)

In [55]:
val f3 = evalExpr(expr3, myEnvironment) // Should be 0.0

f3: Double = 0.0

In [56]:
val f4 = Log(Plus(List(Const(1.0), Mult(List(Const(2.0), Ident("y"))))))
val env = Map("y" -> 1.2)

f4: Log = Log(
  e = Plus(
    eList = List(
      Const(f = 1.0),
      Mult(eList = List(Const(f = 2.0), Ident(s = "y")))
    )
  )
)
env: Map[String, Double] = Map("y" -> 1.2)

In [57]:
val f5 = evalExpr(f4,env )

f5: Double = 1.2237754316221157

We learned how to implement some operations on our inductive data structures.
Now we are ready in the next step to learn about how to write an interpreter and start adding features to the interpreter. But you may have already sensed something important: we have developed the bad habit of writing
lots of code to carry out various operations and using the code as the definition. This works for simple operations which are easy for us to understand. For a more complex beast like a program, we will need a disciplined approach to understanding and writing down the workings of the program outside of code and in a way that can be analyzed by people. We will learn these as operational semantics in the coming lectures.

In [15]:
sealed trait Regex
case object Empty extends Regex
case class Match(s: String) extends Regex
case class Or(r1: Regex, r2: Regex) extends Regex
case class Seq(r1: Regex, r2: Regex) extends Regex
case class Star(r: Regex) extends Regex

(console):3:18 expected ")"
case class Match(String) extends Regex
                 ^